In [1]:
# !pip install h2o pandas requests matplotlib seaborn

import io, os, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import requests

import h2o
from h2o.automl import H2OAutoML

warnings.filterwarnings('ignore')

In [2]:
RAW_CSV_URL = (
    'https://raw.githubusercontent.com/jason12102/MLOps-Final-Project'
    '/main/data_versioning/diabetes.csv'
)

WORK_DIR   = Path('data_versioning')
SCRIPT_DIR = WORK_DIR / 'scripts'
OUTPUT_DIR = Path('h2o_automl_results')
for d in [WORK_DIR, SCRIPT_DIR, OUTPUT_DIR]:
    d.mkdir(exist_ok=True)

TARGET = 'Outcome'
MAX_MODELS = 20
MAX_RUNTIME_SECS = 120
SEED = 42

In [3]:
resp = requests.get(RAW_CSV_URL, timeout=30)
resp.raise_for_status()

raw_csv = WORK_DIR / 'diabetes.csv'
raw_csv.write_bytes(resp.content.rstrip(b'\r\n'))

print(f'Fetched {raw_csv}  ({len(resp.content):,} bytes)')
pd.read_csv(raw_csv).head(3)

Fetched data_versioning\diabetes.csv  (23,875 bytes)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1


In [4]:
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
BMI_BINS   = [float("-inf"), 18.5, 25, 30, float("inf")]
BMI_LABELS = ["Underweight", "Normal", "Overweight", "Obese"]
AGE_BINS   = [float("-inf"), 29, 39, 49, 59, float("inf")]
AGE_LABELS = ["20s", "30s", "40s", "50s", "60+"]

def make_v2(df):
    df = df.copy()
    for col in ZERO_AS_MISSING:
        median = df.loc[df[col] != 0, col].median()
        df.loc[df[col] == 0, col] = median
    return df

def make_v3(df):
    df = df.copy()
    df["BMI_category"] = pd.cut(df["BMI"], bins=BMI_BINS, labels=BMI_LABELS, right=False)
    df["Age_group"]    = pd.cut(df["Age"],  bins=AGE_BINS,  labels=AGE_LABELS)
    df["Glucose_BMI"]  = df["Glucose"] * df["BMI"]
    return df

df_v1 = pd.read_csv(raw_csv)
df_v2 = make_v2(df_v1)
df_v3 = make_v3(df_v2)

dfs = {
    "v1_raw": df_v1,
    "v2_cleaned": df_v2,
    "v3_features": df_v3,
}

In [5]:
import hashlib

DVC_MD5 = {
    "v1_raw":      "f2906818eda8fcfc8f8416557ab1e6df",
    "v2_cleaned":  "44c20ca9c2f305d5cb8ddfd4c8c5c4bd",
    "v3_features": "5ea4cf63f4a3fa89ba570b283e0179eb",
}
DVC_SIZE = {
    "v1_raw":      23873,
    "v2_cleaned":  24321,
    "v3_features": 40838,
}

IMPUTED_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# v1 can be verified exactly — we have the original fetched bytes
v1_bytes    = raw_csv.read_bytes()
v1_md5      = hashlib.md5(v1_bytes).hexdigest()
v1_md5_ok   = v1_md5 == DVC_MD5["v1_raw"]
v1_size_ok  = len(v1_bytes) == DVC_SIZE["v1_raw"]

print("DVC Integrity Check")
print("=" * 60)
print(f"v1_raw  md5  : {'OK' if v1_md5_ok  else 'MISMATCH'}  ({v1_md5[:10]}...)")
print(f"v1_raw  size : {'OK' if v1_size_ok else 'MISMATCH'}  ({len(v1_bytes)} bytes, expected {DVC_SIZE['v1_raw']})")
print()
print("v2/v3: generated in-memory from v1 — byte-exact DVC match")
print("not possible without writing through the original author's")
print("pandas version. Structural checks used instead:")
print()

# Structural checks for v2 and v3
print(f"{'Version':<15} {'Shape':<12} {'New Cols':<35} Zeros in imputed")
print("-" * 95)
for ver, df in dfs.items():
    zeros    = {c: int((df[c] == 0).sum()) for c in IMPUTED_COLS}
    new_cols = [c for c in df.columns if c not in dfs["v1_raw"].columns]
    print(f"{ver:<15} {str(df.shape):<12} {str(new_cols):<35} {zeros}")

print()
# v2 structural: no zeros in imputed cols
v2_zeros_ok = all((dfs["v2_cleaned"][c] == 0).sum() == 0 for c in IMPUTED_COLS)
# v3 structural: expected new columns present
v3_cols_ok  = all(c in dfs["v3_features"].columns for c in ["BMI_category", "Age_group", "Glucose_BMI"])

print(f"v1 DVC hash  : {'PASSED' if v1_md5_ok and v1_size_ok else 'FAILED'}")
print(f"v2 no zeros  : {'PASSED' if v2_zeros_ok else 'FAILED'}")
print(f"v3 new cols  : {'PASSED' if v3_cols_ok  else 'FAILED'}")


DVC Integrity Check
v1_raw  md5  : OK  (f2906818ed...)
v1_raw  size : OK  (23873 bytes, expected 23873)

v2/v3: generated in-memory from v1 — byte-exact DVC match
not possible without writing through the original author's
pandas version. Structural checks used instead:

Version         Shape        New Cols                            Zeros in imputed
-----------------------------------------------------------------------------------------------
v1_raw          (768, 9)     []                                  {'Glucose': 5, 'BloodPressure': 35, 'SkinThickness': 227, 'Insulin': 374, 'BMI': 11}
v2_cleaned      (768, 9)     []                                  {'Glucose': 0, 'BloodPressure': 0, 'SkinThickness': 0, 'Insulin': 0, 'BMI': 0}
v3_features     (768, 12)    ['BMI_category', 'Age_group', 'Glucose_BMI'] {'Glucose': 0, 'BloodPressure': 0, 'SkinThickness': 0, 'Insulin': 0, 'BMI': 0}

v1 DVC hash  : PASSED
v2 no zeros  : PASSED
v3 new cols  : PASSED


In [6]:
n = len(dfs['v1_raw'])
rng = np.random.default_rng(SEED)
test_idx = sorted(rng.choice(n, size=int(n * 0.15), replace=False).tolist())
train_idx = [i for i in range(n) if i not in set(test_idx)]

print(f'Total rows : {n}')
print(f'Train      : {len(train_idx)}')
print(f'Test (held): {len(test_idx)}')

Total rows : 768
Train      : 653
Test (held): 115


In [7]:
h2o.init(nthreads=-1, max_mem_size='4G')
h2o.no_progress()

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,12 hours 18 mins
H2O_cluster_timezone:,America/Chicago
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,2 months and 9 days
H2O_cluster_name:,H2O_from_python_Joshua_qp6fvy
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.016 Gb
H2O_cluster_total_cores:,22
H2O_cluster_allowed_cores:,22
H2O_cluster_status:,"locked, healthy"


In [8]:
def best_val(pairs):
    """pairs is [[threshold, value], ...] — return the value at the best threshold."""
    return max(v for _, v in pairs)

def extract_metrics(perf, model, ver):
    # Accuracy: 1 - minimum mean_per_class_error across thresholds
    acc = 1 - min(v for _, v in perf.mean_per_class_error())
    return {
        'version': ver,
        'model_id': model.model_id,
        'AUC': round(perf.auc(), 4),
        'AUCPR': round(perf.aucpr(), 4),
        'Logloss': round(perf.logloss(), 4),
        'F1': round(best_val(perf.F1()), 4),
        'Accuracy': round(acc, 4),
        'Precision': round(best_val(perf.precision()), 4),
        'Recall': round(best_val(perf.recall()), 4),
    }

results = {}
summary_rows = []

for ver, df in dfs.items():
    print(f'\n{"="*58}')
    print(f'  AutoML -> {ver.upper()}')
    print(f'{"="*58}')

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)

    hf_train = h2o.H2OFrame(df_train)
    hf_test  = h2o.H2OFrame(df_test)
    hf_train[TARGET] = hf_train[TARGET].asfactor()
    hf_test[TARGET]  = hf_test[TARGET].asfactor()

    for cat in ['BMI_category', 'Age_group']:
        if cat in hf_train.columns:
            hf_train[cat] = hf_train[cat].asfactor()
            hf_test[cat]  = hf_test[cat].asfactor()

    features = [c for c in hf_train.columns if c != TARGET]
    print(f'  train={hf_train.nrow}  test={hf_test.nrow}  features({len(features)}): {features}')

    aml = H2OAutoML(
        max_models=MAX_MODELS,
        max_runtime_secs=MAX_RUNTIME_SECS,
        seed=SEED,
        project_name=f'diabetes_{ver}',
        sort_metric='AUC',
        balance_classes=True,
    )
    aml.train(x=features, y=TARGET, training_frame=hf_train)

    perf    = aml.leader.model_performance(hf_test)
    metrics = extract_metrics(perf, aml.leader, ver)
    summary_rows.append(metrics)
    results[ver] = {'aml': aml, 'perf': perf, 'hf_test': hf_test}

    lb = aml.leaderboard.as_data_frame()
    lb.to_csv(OUTPUT_DIR / f'leaderboard_{ver}.csv', index=False)

    print(f'\n  Best model : {aml.leader.model_id}')
    for k in ['AUC','AUCPR','F1','Accuracy','Precision','Recall','Logloss']:
        print(f'    {k:<10}: {metrics[k]}')
    print(f'\n  Top 5 Leaderboard:')
    display(lb.head(5))


  AutoML -> V1_RAW
  train=653  test=115  features(8): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

23:57:16.269: AutoML: XGBoost is not available; skipping it.
11:32:47.581: New models will be added to existing leaderboard diabetes_v1_raw@@Outcome (leaderboard frame=null) with already 15 models.
11:32:47.581: AutoML: XGBoost is not available; skipping it.
12:16:14.421: New models will be added to existing leaderboard diabetes_v1_raw@@Outcome (leaderboard frame=null) with already 32 models.
12:16:14.422: AutoML: XGBoost is not available; skipping it.


  Best model : GBM_11_AutoML_7_20260521_121614
    AUC       : 0.7756
    AUCPR     : 0.6417
    F1        : 0.6471
    Accuracy  : 0.7495
    Precision : 1.0
    Recall    : 1.0
    Logloss   : 0.5196

  Top 5 Leaderboard:


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,GBM_11_AutoML_7_20260521_121614,0.843818,0.465955,0.741230,0.209769,0.392169,0.153797
1,GBM_6_AutoML_4_20260521_113247,0.843818,0.465955,0.741230,0.209769,0.392169,0.153797
2,GBM_1_AutoML_1_20260520_235716,0.843818,0.465955,0.741230,0.209769,0.392169,0.153797
3,DeepLearning_grid_4_AutoML_7_20260521_121614_m...,0.843186,0.505562,0.721228,0.216597,0.402075,0.161664
4,GBM_10_AutoML_4_20260521_113247,0.838270,0.482276,0.725982,0.226363,0.398067,0.158457



  AutoML -> V2_CLEANED
  train=653  test=115  features(8): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

23:59:24.728: AutoML: XGBoost is not available; skipping it.
11:36:22.924: New models will be added to existing leaderboard diabetes_v2_cleaned@@Outcome (leaderboard frame=null) with already 15 models.
11:36:22.924: AutoML: XGBoost is not available; skipping it.
12:19:47.66: New models will be added to existing leaderboard diabetes_v2_cleaned@@Outcome (leaderboard frame=null) with already 31 models.
12:19:47.66: AutoML: XGBoost is not available; skipping it.


  Best model : DeepLearning_grid_2_AutoML_5_20260521_113622_model_2
    AUC       : 0.7658
    AUCPR     : 0.5712
    F1        : 0.6
    Accuracy  : 0.7162
    Precision : 0.8333
    Recall    : 1.0
    Logloss   : 0.6799

  Top 5 Leaderboard:


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,DeepLearning_grid_2_AutoML_5_20260521_113622_m...,0.856618,0.475554,0.725058,0.203415,0.390345,0.152369
1,DeepLearning_grid_4_AutoML_8_20260521_121947_m...,0.855802,0.478816,0.727356,0.210156,0.391577,0.153332
2,GBM_15_AutoML_8_20260521_121947,0.843222,0.472892,0.742462,0.220815,0.396080,0.156879
3,GBM_10_AutoML_5_20260521_113622,0.843222,0.472892,0.742462,0.220815,0.396080,0.156879
4,GBM_5_AutoML_2_20260520_235924,0.843222,0.472892,0.742462,0.220815,0.396080,0.156879



  AutoML -> V3_FEATURES
  train=653  test=115  features(11): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'BMI_category', 'Age_group', 'Glucose_BMI']

00:01:29.318: AutoML: XGBoost is not available; skipping it.
11:40:02.498: New models will be added to existing leaderboard diabetes_v3_features@@Outcome (leaderboard frame=null) with already 15 models.
11:40:02.498: AutoML: XGBoost is not available; skipping it.
12:23:20.672: New models will be added to existing leaderboard diabetes_v3_features@@Outcome (leaderboard frame=null) with already 32 models.
12:23:20.672: AutoML: XGBoost is not available; skipping it.


  Best model : DeepLearning_grid_4_AutoML_9_20260521_122320_model_2
    AUC       : 0.7658
    AUCPR     : 0.5623
    F1        : 0.6216
    Accuracy  : 0.7333
    Precision : 0.8571
    Recall    : 1.0
    Logloss   : 0.8592

  Top 5 Leaderboard:


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,DeepLearning_grid_4_AutoML_9_20260521_122320_m...,0.855481,0.490209,0.732809,0.215929,0.394369,0.155527
1,GLM_1_AutoML_3_20260521_00129,0.848801,0.465261,0.732319,0.220315,0.390597,0.152566
2,GLM_2_AutoML_6_20260521_114002,0.848801,0.465261,0.732319,0.220315,0.390597,0.152566
3,GLM_3_AutoML_9_20260521_122320,0.848801,0.465261,0.732319,0.220315,0.390597,0.152566
4,GBM_1_AutoML_3_20260521_00129,0.847072,0.460367,0.735891,0.207744,0.389764,0.151916
